In [11]:
import polars as pl
import altair as alt
import numpy as np

alt.data_transformers.disable_max_rows()

DATA_DIR = r"C:\Users\ralex\Documents\GitHub\visualizacao-lab3\DADOS"

RAZOES = {
    "A": "Gratuidade",
    "B": "Preço da mensalidade",
    "C": "Proximidade da residência",
    "D": "Proximidade do trabalho",
    "E": "Facilidade de acesso",
    "F": "Qualidade/ reputação",
    "G": "Única com aprovação",
    "H": "Possibilidade de bolsa",
    "I": "Outro motivo",
}

MIN_RESPOSTAS = 100

In [12]:
arq3 = pl.read_csv(
    f"{DATA_DIR}/microdados2023_arq3.txt",
    separator=";", encoding="latin1",
    columns=["CO_CURSO", "NT_GER"],
    null_values=["", " "],
    infer_schema=False,
).with_columns([
    pl.col("CO_CURSO").cast(pl.Int64),
    pl.col("NT_GER").str.replace(",", ".").cast(pl.Float64, strict=False),
])

arq32 = pl.read_csv(
    f"{DATA_DIR}/microdados2023_arq32.txt",
    separator=";", encoding="latin1",
    columns=["CO_CURSO", "QE_I26"],
    null_values=["", " "],
).with_columns(pl.col("CO_CURSO").cast(pl.Int64))

# Nota média e número de alunos por curso
media_por_curso = (
    arq3
    .filter(pl.col("NT_GER").is_not_null() & (pl.col("NT_GER") > 0))
    .group_by("CO_CURSO")
    .agg([
        pl.col("NT_GER").mean().alias("NT_GER_media"),
        pl.len().alias("n_alunos"),
    ])
)

respostas = (
    arq32
    .filter(pl.col("QE_I26").is_not_null())
    .with_columns(pl.col("QE_I26").replace(RAZOES).alias("Razao"))
)

cursos_validos = (
    respostas
    .group_by("CO_CURSO")
    .agg(pl.len().alias("n_total"))
    .filter(pl.col("n_total") >= MIN_RESPOSTAS)
)

moda_por_curso = (
    respostas
    .join(cursos_validos.select("CO_CURSO"), on="CO_CURSO", how="inner")
    .group_by(["CO_CURSO", "Razao"])
    .agg(pl.len().alias("n"))
    .group_by("CO_CURSO")
    .agg(
        pl.col("Razao")
        .sort_by(["n", "Razao"], descending=[True, False])
        .first()
    )
)

# Um curso por linha com nota média, número de alunos e razão predominante
df = (
    moda_por_curso
    .join(media_por_curso, on="CO_CURSO", how="inner")
    .filter(pl.col("NT_GER_media").is_not_null())
    .sort("CO_CURSO")
    .to_pandas()
)

print(f"Cursos: {len(df):,}")
print(f"Alunos (total): {df['n_alunos'].sum():,}")
df.head()

Cursos: 577
Alunos (total): 105,004


,CO_CURSO,Razao,NT_GER_media,n_alunos
0,132,Qualidade/ reputação,68.927407,135
1,389,Gratuidade,63.065000,120
2,619,Qualidade/ reputação,47.561818,110
3,685,Qualidade/ reputação,66.400749,267
4,864,Qualidade/ reputação,63.224752,101


In [19]:
import pandas as pd

# Ordem pela mediana da nota, maior → menor
ordem_nota = (
    df.groupby("Razao")["NT_GER_media"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)
if "Outro motivo" in ordem_nota:
    ordem_nota.remove("Outro motivo")
    ordem_nota.append("Outro motivo")

rng = np.random.default_rng(41)
df["jitter"] = rng.uniform(-0.45, 0.45, size=len(df))

df_median = (
    df.groupby("Razao")["NT_GER_media"]
    .median()
    .reset_index()
    .rename(columns={"NT_GER_media": "mediana"})
)

bubble = (
    alt.Chart(df)
    .mark_circle(opacity=0.45)
    .encode(
        x=alt.X("Razao:N", sort=ordem_nota, title=None,
                axis=alt.Axis(
                    labelAngle=0,
                    labelLimit=100,
                    labelLineHeight=12,
                    labelExpr="split(datum.label, ' ')",
                )),
        xOffset=alt.XOffset("jitter:Q", scale=alt.Scale(domain=[-0.5, 0.5])),
        y=alt.Y("NT_GER_media:Q", title="Nota média do curso",
                scale=alt.Scale(zero=True)),
        size=alt.Size("n_alunos:Q", title="Nº de alunos",
                      scale=alt.Scale(range=[20, 800]),
                      legend=alt.Legend(values=[100, 500, 1000, 1500, 2000, 2500])),
        color=alt.Color("Razao:N", legend=None),
        tooltip=[
            "Razao",
            alt.Tooltip("NT_GER_media:Q", format=".1f", title="Nota média"),
            alt.Tooltip("n_alunos:Q", format=",", title="Nº de alunos"),
        ],
    )
)

mediana_layer = (
    alt.Chart(df_median)
    .mark_tick(thickness=2, size=40, color="black", opacity=0.85)
    .encode(
        x=alt.X("Razao:N", sort=ordem_nota),
        y=alt.Y("mediana:Q"),
        tooltip=["Razao", alt.Tooltip("mediana:Q", format=".1f", title="Mediana")],
    )
)

label_layer = (
    alt.Chart(df_median)
    .mark_text(dx=25, fontSize=11, fontWeight="bold", color="black",
               align="left", baseline="middle")
    .encode(
        x=alt.X("Razao:N", sort=ordem_nota),
        y=alt.Y("mediana:Q"),
        text=alt.Text("mediana:Q", format=".1f"),
    )
)

(bubble + mediana_layer + label_layer).properties(
    title=alt.TitleParams(
        "Nota média do curso (NT_GER) × Razão predominante de escolha da instituição (QE_I26)",
        subtitle="Tamanho do círculo = número de alunos no curso  |  Traço = mediana do grupo",
    ),
    width=700, height=300,
)

alt.LayerChart(...)